# Joins — Broadcast Join Control

Compare three join strategies for **order_items ⋈ sellers**: Spark default, forced sort-merge (`hint("merge")`), and explicit `broadcast()`. Record `explain()` output, detected strategy, and shuffle presence.

*Databricks Free blocks `spark.conf` broadcast toggles — hints are used instead.*

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.joins.broadcast_control as bc_module
importlib.reload(bc_module)

from src.joins.business_questions import SilverJoinTables
from src.joins.broadcast_control import (
    join_items_sellers_default,
    join_items_sellers_sort_merge,
    join_items_sellers_broadcast,
    analyze_join_variant,
    run_broadcast_join_comparison,
)
from src.spark_performance.execution_plans import capture_explain
from src.ingestion.idempotent_loader import save_run_report_to_volume

tables = SilverJoinTables()
print("Loaded from:", bc_module.__file__)
print("Items:", spark.table(tables.order_items).count(), "| Sellers:", spark.table(tables.sellers).count())

In [ ]:
default_df = join_items_sellers_default(spark, tables)
merge_df = join_items_sellers_sort_merge(spark, tables)
broadcast_df = join_items_sellers_broadcast(spark, tables)

default_result = analyze_join_variant("default", "Spark default", default_df)
merge_result = analyze_join_variant("sort_merge", "hint(merge)", merge_df)
broadcast_result = analyze_join_variant("broadcast", "broadcast(sellers)", broadcast_df)

print("=== DEFAULT ===")
print(capture_explain(default_df))
print(default_result)

print("=== SORT-MERGE (hint) ===")
print(capture_explain(merge_df))
print(merge_result)

print("=== BROADCAST ===")
print(capture_explain(broadcast_df))
print(broadcast_result)

In [ ]:
import json

report = run_broadcast_join_comparison(spark, tables)
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/joins_broadcast_control.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)